# Datathon 2026 — Phần 1: Câu hỏi Trắc nghiệm
Chạy từng cell theo thứ tự. Đáp án sẽ hiện ở cuối mỗi cell.

## Load dữ liệu
Sửa `DATA_DIR` thành đường dẫn thư mục chứa các file CSV của bạn.

In [16]:
import pandas as pd
import numpy as np
import os

# Tìm đường dẫn thư mục Data (lên 1 level từ Multiple Choice)
notebook_dir = os.path.dirname(os.path.abspath("./solve_mcq.ipynb"))
data_dir = os.path.join(os.path.dirname(notebook_dir), "Data")

# Hoặc sử dụng đường dẫn absolute
DATA_DIR = r"C:\Users\Admin\Desktop\UT-UIT\Datathon_Contest_UITUT\Data\\"

orders    = pd.read_csv(DATA_DIR + "orders.csv",     parse_dates=["order_date"])
products  = pd.read_csv(DATA_DIR + "products.csv")
returns   = pd.read_csv(DATA_DIR + "returns.csv")
web       = pd.read_csv(DATA_DIR + "web_traffic.csv")
items     = pd.read_csv(DATA_DIR + "order_items.csv")
customers = pd.read_csv(DATA_DIR + "customers.csv")
geo       = pd.read_csv(DATA_DIR + "geography.csv")
payments  = pd.read_csv(DATA_DIR + "payments.csv")

C:\Users\Admin\AppData\Local\Temp\ipykernel_4968\718636663.py:16: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  items     = pd.read_csv(DATA_DIR + "order_items.csv")


---
## Q1 — Median inter-order gap
**Câu hỏi:** Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp xấp xỉ là bao nhiêu?

In [5]:
# Sắp xếp theo customer_id rồi theo ngày đặt hàng
o = orders.sort_values(["customer_id", "order_date"])

# Tính gap (số ngày) giữa 2 đơn liên tiếp của cùng 1 khách
# .diff() lấy hiệu với dòng trước trong cùng nhóm
# .dt.days chuyển timedelta → số nguyên
o["gap"] = o.groupby("customer_id")["order_date"].diff().dt.days

# Loại bỏ NaN (đơn đầu tiên của mỗi khách không có gap)
multi = o[o["gap"].notna()]

median_gap = multi["gap"].median()
print(f"Median gap = {median_gap:.1f} ngày")

# So sánh với các đáp án
options = {"A": 30, "B": 90, "C": 180, "D": 365}
q1 = min(options, key=lambda x: abs(median_gap - options[x]))
print(f"\n>>> ĐÁP ÁN Q1: {q1}")

Median gap = 144.0 ngày

>>> ĐÁP ÁN Q1: C


---
## Q2 — Segment có gross margin cao nhất
**Câu hỏi:** Phân khúc sản phẩm nào có tỷ suất lợi nhuận gộp trung bình cao nhất?

In [6]:
# Tính margin cho từng sản phẩm
products["margin"] = (products["price"] - products["cogs"]) / products["price"]

# Trung bình margin theo segment
ans = products.groupby("segment")["margin"].mean().sort_values(ascending=False)
print("Margin trung bình theo segment:")
print(ans.round(4).to_string())

seg_map = {"Premium": "A", "Performance": "B", "Activewear": "C", "Standard": "D"}
top_seg = ans.idxmax()
q2 = seg_map.get(top_seg, top_seg)
print(f"\n>>> ĐÁP ÁN Q2: {q2}  ({top_seg})")

Margin trung bình theo segment:
segment
Standard       0.3134
Premium        0.2854
All-weather    0.2842
Activewear     0.2656
Performance    0.2636
Balanced       0.2580
Trendy         0.2408
Everyday       0.2363

>>> ĐÁP ÁN Q2: D  (Standard)


---
## Q3 — Return reason phổ biến nhất trong Streetwear
**Câu hỏi:** Trong các bản ghi trả hàng của danh mục Streetwear, lý do nào xuất hiện nhiều nhất?

In [7]:
# Join để lấy category của sản phẩm bị trả
merged3 = returns.merge(products[["product_id", "category"]], on="product_id")

# Lọc chỉ Streetwear
sw = merged3[merged3["category"] == "Streetwear"]

# Đếm lý do
reason_counts = sw["return_reason"].value_counts()
print("Số lần xuất hiện của từng lý do trả hàng (Streetwear):")
print(reason_counts.to_string())

top_reason = reason_counts.idxmax()
reason_map = {"defective": "A", "wrong_size": "B", "changed_mind": "C", "not_as_described": "D"}
q3 = reason_map.get(top_reason, top_reason)
print(f"\n>>> ĐÁP ÁN Q3: {q3}  ({top_reason})")

Số lần xuất hiện của từng lý do trả hàng (Streetwear):
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159

>>> ĐÁP ÁN Q3: B  (wrong_size)


---
## Q4 — Traffic source có bounce rate thấp nhất
**Câu hỏi:** Nguồn truy cập nào có tỷ lệ thoát trung bình thấp nhất?

In [8]:
# Trung bình bounce rate theo traffic source
ans4 = web.groupby("traffic_source")["bounce_rate"].mean().sort_values()
print("Bounce rate trung bình theo source (thấp → cao):")
print(ans4.round(4).to_string())

top_src = ans4.idxmin()
src_map = {"organic_search": "A", "paid_search": "B", "email_campaign": "C", "social_media": "D"}
q4 = src_map.get(top_src, top_src)
print(f"\n>>> ĐÁP ÁN Q4: {q4}  ({top_src})")

Bounce rate trung bình theo source (thấp → cao):
traffic_source
email_campaign    0.0045
social_media      0.0045
paid_search       0.0045
referral          0.0045
organic_search    0.0045
direct            0.0045

>>> ĐÁP ÁN Q4: C  (email_campaign)


---
## Q5 — % order_items có áp dụng khuyến mãi
**Câu hỏi:** Tỷ lệ % các dòng trong order_items có promo_id không null?

In [9]:
# .notna() → True nếu có promo, False nếu không
# .mean() → tỷ lệ True = % có promo
pct = items["promo_id"].notna().mean() * 100
print(f"% dòng có promo_id: {pct:.2f}%")

if pct < 18:    q5 = "A"
elif pct < 31:  q5 = "B"
elif pct < 46:  q5 = "C"
else:           q5 = "D"
print(f"\n>>> ĐÁP ÁN Q5: {q5}")

% dòng có promo_id: 38.66%

>>> ĐÁP ÁN Q5: C


---
## Q6 — Age group có avg orders/customer cao nhất
**Câu hỏi:** Nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất?

In [10]:
# Đếm số đơn của từng khách hàng
order_counts = orders.groupby("customer_id").size().reset_index(name="n_orders")

# Lọc khách có age_group, rồi join số đơn vào
cust = customers[customers["age_group"].notna()].merge(
    order_counts, on="customer_id", how="left"
)
# Khách chưa có đơn nào → fillna(0)
cust["n_orders"] = cust["n_orders"].fillna(0)

# Trung bình số đơn theo nhóm tuổi
ans6 = cust.groupby("age_group")["n_orders"].mean().sort_values(ascending=False)
print("Avg orders/customer theo age_group:")
print(ans6.round(3).to_string())

top_age = ans6.idxmax()
age_map = {"55+": "A", "25-34": "B", "35-44": "C", "45-54": "D"}
q6 = age_map.get(top_age, top_age)
print(f"\n>>> ĐÁP ÁN Q6: {q6}  ({top_age})")

Avg orders/customer theo age_group:
age_group
55+      5.407
45-54    5.357
35-44    5.337
25-34    5.245
18-24    5.227

>>> ĐÁP ÁN Q6: A  (55+)


---
## Q7 — Region có tổng revenue cao nhất
**Câu hỏi:** Vùng nào tạo ra tổng doanh thu cao nhất?

In [11]:
# Tính revenue từng dòng sản phẩm: quantity × unit_price − discount
order_rev = items.copy()
order_rev["line_rev"] = order_rev["quantity"] * order_rev["unit_price"] - order_rev["discount_amount"]

# Cộng lại theo đơn hàng
order_rev = order_rev.groupby("order_id")["line_rev"].sum().reset_index()

# Join orders → geography để lấy region
merged7 = (
    orders[["order_id", "zip"]]
    .merge(geo[["zip", "region"]], on="zip")
    .merge(order_rev, on="order_id")
)

# Tổng revenue theo region
ans7 = merged7.groupby("region")["line_rev"].sum().sort_values(ascending=False)
print("Tổng revenue theo region:")
print(ans7.apply(lambda x: f"{x:,.0f}").to_string())

top_region = ans7.idxmax()
reg_map = {"West": "A", "Central": "B", "East": "C"}
q7 = reg_map.get(top_region, "D")
print(f"\n>>> ĐÁP ÁN Q7: {q7}  ({top_region})")

Tổng revenue theo region:
region
East       7,291,150,819
Central    4,719,491,268
West       3,670,227,178

>>> ĐÁP ÁN Q7: C  (East)


---
## Q8 — Payment method phổ biến nhất trong đơn cancelled

In [12]:
# Lọc đơn cancelled
cancelled = orders[orders["order_status"] == "cancelled"]

# Đếm payment method
pay_counts = cancelled["payment_method"].value_counts()
print("Số lần dùng mỗi payment method trong đơn cancelled:")
print(pay_counts.to_string())

top_pay = pay_counts.idxmax()
pay_map = {"credit_card": "A", "cod": "B", "paypal": "C", "bank_transfer": "D"}
q8 = pay_map.get(top_pay, top_pay)
print(f"\n>>> ĐÁP ÁN Q8: {q8}  ({top_pay})")

Số lần dùng mỗi payment method trong đơn cancelled:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535

>>> ĐÁP ÁN Q8: A  (credit_card)


---
## Q9 — Size có return rate cao nhất
**Câu hỏi:** Trong 4 kích thước S/M/L/XL, size nào có tỷ lệ trả hàng cao nhất?

In [13]:
# Join cả 2 bảng với products để lấy cột size
items_s = items.merge(products[["product_id", "size"]], on="product_id")
ret_s   = returns.merge(products[["product_id", "size"]], on="product_id")

sizes = ["S", "M", "L", "XL"]

# Tổng số lần mua theo size
total   = items_s[items_s["size"].isin(sizes)].groupby("size").size()
# Số lần trả theo size
ret_cnt = ret_s[ret_s["size"].isin(sizes)].groupby("size").size()

# Tỷ lệ return
rate = (ret_cnt / total).sort_values(ascending=False)
print("Return rate theo size:")
print(rate.round(4).to_string())

top_size = rate.idxmax()
size_map = {"S": "A", "M": "B", "L": "C", "XL": "D"}
q9 = size_map.get(top_size, top_size)
print(f"\n>>> ĐÁP ÁN Q9: {q9}  ({top_size})")

Return rate theo size:
size
S     0.0565
L     0.0562
M     0.0557
XL    0.0552

>>> ĐÁP ÁN Q9: A  (S)


---
## Q10 — Installment plan có avg payment value cao nhất

In [14]:
# Trung bình payment_value theo số kỳ trả góp
ans10 = payments.groupby("installments")["payment_value"].mean().sort_values(ascending=False)
print("Avg payment value theo số kỳ:")
print(ans10.round(2).to_string())

top_inst = ans10.idxmax()
inst_map = {1: "A", 3: "B", 6: "C", 12: "D"}
q10 = inst_map.get(top_inst, str(top_inst))
print(f"\n>>> ĐÁP ÁN Q10: {q10}  ({top_inst} kỳ)")

Avg payment value theo số kỳ:
installments
6     24446.65
3     24399.64
12    24245.77
1     24113.27
2       708.47

>>> ĐÁP ÁN Q10: C  (6 kỳ)


---
## Tóm tắt tất cả đáp án

In [15]:
print("╔══════════════════════════════╗")
print("║     TÓM TẮT ĐÁP ÁN MCQ      ║")
print("╠══════════════════════════════╣")
answers = {"Q1": q1, "Q2": q2, "Q3": q3, "Q4": q4, "Q5": q5,
           "Q6": q6, "Q7": q7, "Q8": q8, "Q9": q9, "Q10": q10}
for q, a in answers.items():
    print(f"║  {q:<4} →  {a}                        ║")
print("╚══════════════════════════════╝")

╔══════════════════════════════╗
║     TÓM TẮT ĐÁP ÁN MCQ      ║
╠══════════════════════════════╣
║  Q1   →  C                        ║
║  Q2   →  D                        ║
║  Q3   →  B                        ║
║  Q4   →  C                        ║
║  Q5   →  C                        ║
║  Q6   →  A                        ║
║  Q7   →  C                        ║
║  Q8   →  A                        ║
║  Q9   →  A                        ║
║  Q10  →  C                        ║
╚══════════════════════════════╝
